# Самостоятельное задание — пример решения

## Вариант 30. Сотовая связь — тарифные планы

Это **образцовое решение** одного из 30 вариантов самостоятельного задания. Цель файла — показать, как должна выглядеть полная сдача: описание модели, код с инкапсуляцией и наследованием, контейнер с осмысленными методами, демонстрация и обработка исключений.


## Описание модели

**Диаграмма классов словами:**

*  **`Tariff`** — базовый класс. Поля: `_name`, `_monthly_fee`, `_minutes`, `_gigabytes`. Доступ через свойства с валидацией. Метод `effective_fee()` возвращает реально списываемую сумму (для базового тарифа = абонплата).
*  **`FamilyTariff(Tariff)`** — наследник. Добавляет `_numbers_count` (≥ 2) и `_discount` (0–100 %). Переопределяет `effective_fee()`: суммарная плата за все номера со скидкой. Переопределяет `__str__`.
*  **`Operator`** — контейнер-агрегат. Хранит список тарифов. Методы: `add_tariff`, `remove_tariff`, `find_tariff`, `affordable`, `sorted_by_price`, `summary`. Поддерживает `__len__` и `__iter__` — это понадобится для заполнения `Listbox`/`Treeview` в GUI.

**Привязка к будущему GUI:**

| Элемент GUI | Метод модели |
|---|---|
| Кнопка «Добавить тариф» в форме создания | `operator.add_tariff(Tariff(...))` или `FamilyTariff(...)` |
| Кнопка «Удалить выбранный» в таблице | `operator.remove_tariff(selected_name)` |
| Поле поиска + кнопка «Найти» | `operator.find_tariff(query)` |
| Слайдер «Макс. цена» + чекбокс «Только доступные» | `operator.affordable(max_fee)` |
| Заголовок столбца «Цена» (клик = сортировка) | `operator.sorted_by_price(descending=...)` |
| Панель статистики внизу окна | `operator.summary()` → заполнить `Label`-ы |
| Заполнение `Listbox`/`Treeview` при открытии окна | `for t in operator: tree.insert(..., str(t))` |
| Поля ввода в форме создания тарифа | проверяются сеттерами `name`, `monthly_fee`, `minutes`, `gigabytes`, `numbers_count`, `discount` — некорректный ввод поднимет `TypeError`/`ValueError`, которые GUI поймает и покажет `messagebox.showerror` |


## 1. Базовый класс `Tariff`

Поля хранятся как `_имя`, доступ — только через свойства с валидацией типа и диапазона. Инициализация в `__init__` идёт **через сеттеры** (`self.name = name`, а не `self._name = name`), поэтому валидация срабатывает уже при создании объекта.


In [ ]:
from typing import Optional


class Tariff:
    """
    Базовый тарифный план.

    Поля (все хранятся как _имя, доступ — через свойства с валидацией):
      _name           — название тарифа (непустая строка, ключ уникальности)
      _monthly_fee    — абонентская плата в рублях (число >= 0)
      _minutes        — количество минут в пакете (целое >= 0)
      _gigabytes      — количество ГБ интернета в пакете (число >= 0)
    """

    def __init__(self, name: str, monthly_fee: float,
                 minutes: int, gigabytes: float):
        # Инициализация ЧЕРЕЗ сеттеры — чтобы валидация сработала
        # уже при создании объекта, а не где-то потом.
        self.name = name
        self.monthly_fee = monthly_fee
        self.minutes = minutes
        self.gigabytes = gigabytes

    # ---- name ---------------------------------------------------------
    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise TypeError("Название тарифа должно быть строкой.")
        if not value.strip():
            raise ValueError("Название тарифа не может быть пустым.")
        self._name = value.strip()

    # ---- monthly_fee --------------------------------------------------
    @property
    def monthly_fee(self) -> float:
        return self._monthly_fee

    @monthly_fee.setter
    def monthly_fee(self, value):
        # bool — подкласс int в Python, явно отсекаем, чтобы True не
        # проходило как число.
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Абонентская плата должна быть числом.")
        if value < 0:
            raise ValueError("Абонентская плата не может быть отрицательной.")
        self._monthly_fee = float(value)

    # ---- minutes ------------------------------------------------------
    @property
    def minutes(self) -> int:
        return self._minutes

    @minutes.setter
    def minutes(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("Количество минут должно быть целым числом.")
        if value < 0:
            raise ValueError("Количество минут не может быть отрицательным.")
        self._minutes = value

    # ---- gigabytes ----------------------------------------------------
    @property
    def gigabytes(self) -> float:
        return self._gigabytes

    @gigabytes.setter
    def gigabytes(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Количество ГБ должно быть числом.")
        if value < 0:
            raise ValueError("Количество ГБ не может быть отрицательным.")
        self._gigabytes = float(value)

    # ---- Бизнес-логика тарифа -----------------------------------------
    def effective_fee(self) -> float:
        """
        Реально списываемая с абонента сумма за месяц.
        У базового тарифа равна абонплате; в производных может
        переопределяться (полиморфизм).
        """
        return self._monthly_fee

    def __str__(self) -> str:
        # Эта строка позже будет показываться в Listbox/Treeview GUI.
        return (f"[Тариф] {self._name}: {self._monthly_fee:.0f} ₽/мес, "
                f"{self._minutes} мин, {self._gigabytes:g} ГБ")

## 2. Производный класс `FamilyTariff(Tariff)`

Добавляет два новых поля и **переопределяет** `effective_fee()` — это иллюстрация полиморфизма. Благодаря этому контейнер сможет одинаково работать со всеми тарифами через единый интерфейс, не делая `isinstance`-проверок.


In [ ]:
class FamilyTariff(Tariff):
    """
    Семейный тариф: объединяет несколько номеров одной семьи и даёт скидку
    от суммарной абонентской платы за все номера.

    Дополнительные поля:
      _numbers_count  — количество объединяемых номеров (целое >= 2)
      _discount       — скидка в процентах (число в диапазоне [0, 100])
    """

    def __init__(self, name: str, monthly_fee: float,
                 minutes: int, gigabytes: float,
                 numbers_count: int, discount: float):
        super().__init__(name, monthly_fee, minutes, gigabytes)
        self.numbers_count = numbers_count
        self.discount = discount

    # ---- numbers_count ------------------------------------------------
    @property
    def numbers_count(self) -> int:
        return self._numbers_count

    @numbers_count.setter
    def numbers_count(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("Количество номеров должно быть целым числом.")
        if value < 2:
            raise ValueError("Семейный тариф требует минимум 2 номера.")
        self._numbers_count = value

    # ---- discount -----------------------------------------------------
    @property
    def discount(self) -> float:
        return self._discount

    @discount.setter
    def discount(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Скидка должна быть числом.")
        if not (0 <= value <= 100):
            raise ValueError("Скидка должна быть в диапазоне [0, 100] %.")
        self._discount = float(value)

    # ---- Переопределение бизнес-логики --------------------------------
    def effective_fee(self) -> float:
        """
        Суммарная плата семьи: (абонплата × количество номеров) со скидкой.
        Именно эта сумма списывается у владельца семейного аккаунта.
        """
        total = self._monthly_fee * self._numbers_count
        return total * (1 - self._discount / 100)

    def __str__(self) -> str:
        return (f"[Семейный] {self._name}: {self._monthly_fee:.0f} ₽/мес × "
                f"{self._numbers_count} ном., скидка {self._discount:g}%, "
                f"итого {self.effective_fee():.0f} ₽/мес "
                f"(пакет: {self._minutes} мин, {self._gigabytes:g} ГБ)")

## 3. Класс-контейнер `Operator`

Все методы **возвращают** результат (объект, список, словарь, `None`), а не печатают его. Это и есть будущий «бэкенд» GUI: завтра обработчики кнопок будут просто вызывать эти методы и помещать результаты в виджеты.


In [ ]:
class Operator:
    """
    Линейка тарифов оператора связи.

    Публичные методы — это будущие "обработчики кнопок" в GUI:
      add_tariff(tariff)              — кнопка «Добавить тариф»
      remove_tariff(name)             — кнопка «Удалить тариф»
      find_tariff(name)               — поиск по названию (для формы редактирования)
      affordable(max_fee)             — кнопка «Показать тарифы до N ₽»
      summary()                       — панель «Статистика линейки»
      sorted_by_price()               — сортировка таблицы по цене
      __iter__ / __len__              — для наполнения Listbox / Treeview
    """

    def __init__(self):
        # Внутреннее хранилище. Через прямой доступ к _tariffs снаружи
        # не работаем — только через методы.
        self._tariffs: list[Tariff] = []

    # ---- Добавление с проверкой уникальности --------------------------
    def add_tariff(self, tariff: Tariff) -> None:
        """Запустить новый тариф в линейку. Уникальность — по названию."""
        if not isinstance(tariff, Tariff):
            raise TypeError("В линейку можно добавлять только объекты Tariff "
                            "или его наследников.")
        # Сравниваем регистронезависимо — пользователь GUI может ввести
        # 'мини' и 'Мини' и ожидать, что это одно и то же.
        if any(t.name.lower() == tariff.name.lower() for t in self._tariffs):
            raise ValueError(f"Тариф с названием '{tariff.name}' уже существует.")
        self._tariffs.append(tariff)

    # ---- Удаление по ключу --------------------------------------------
    def remove_tariff(self, name: str) -> Tariff:
        """
        Вывести тариф из линейки. Возвращает удалённый объект,
        чтобы GUI мог показать сообщение «Удалён тариф ...».
        Если не найден — ValueError.
        """
        if not isinstance(name, str):
            raise TypeError("Имя тарифа должно быть строкой.")
        for i, t in enumerate(self._tariffs):
            if t.name.lower() == name.lower():
                return self._tariffs.pop(i)
        raise ValueError(f"Тариф '{name}' не найден.")

    # ---- Поиск по ключу -----------------------------------------------
    def find_tariff(self, name: str) -> Optional[Tariff]:
        """Найти тариф по названию. Возвращает объект или None."""
        if not isinstance(name, str):
            raise TypeError("Имя тарифа должно быть строкой.")
        for t in self._tariffs:
            if t.name.lower() == name.lower():
                return t
        return None

    # ---- Фильтрация ---------------------------------------------------
    def affordable(self, max_fee: float) -> list:
        """
        Вернуть список тарифов с эффективной платой <= max_fee.
        Сравниваем именно effective_fee(), чтобы семейные тарифы
        участвовали в фильтре по их реальной итоговой стоимости.
        """
        if isinstance(max_fee, bool) or not isinstance(max_fee, (int, float)):
            raise TypeError("Порог стоимости должен быть числом.")
        if max_fee < 0:
            raise ValueError("Порог стоимости не может быть отрицательным.")
        return [t for t in self._tariffs if t.effective_fee() <= max_fee]

    # ---- Сортировка ---------------------------------------------------
    def sorted_by_price(self, descending: bool = False) -> list:
        """
        Возвращает НОВЫЙ отсортированный список (исходный порядок
        в линейке не меняется — это важно, потому что в GUI
        пользователь может потом снять сортировку).
        """
        return sorted(self._tariffs,
                      key=lambda t: t.effective_fee(),
                      reverse=descending)

    # ---- Сводная статистика -------------------------------------------
    def summary(self) -> dict:
        """
        Сводка для информационной панели GUI.
        Возвращает dict, а не печатает — GUI сам решит, как отобразить.
        """
        count = len(self._tariffs)
        if count == 0:
            return {"count": 0, "avg_gigabytes": 0.0,
                    "avg_fee": 0.0, "min_fee": None, "max_fee": None}

        fees = [t.effective_fee() for t in self._tariffs]
        gbs = [t.gigabytes for t in self._tariffs]
        return {
            "count": count,
            "avg_gigabytes": sum(gbs) / count,
            "avg_fee": sum(fees) / count,
            "min_fee": min(fees),
            "max_fee": max(fees),
        }

    # ---- Протоколы --------------------------------------------------
    def __len__(self) -> int:
        return len(self._tariffs)

    def __iter__(self):
        # Позволит в GUI делать: for t in operator: listbox.insert(END, str(t))
        return iter(self._tariffs)

## 4. Демонстрация работы модели

Создаём 7 тарифов (5 базовых + 2 семейных), демонстрируем **каждый** публичный метод контейнера и обрабатываем 4 разных исключения.


In [ ]:
op = Operator()

print("=" * 60)
print("1. Добавление тарифов в линейку")
print("=" * 60)

# 5 базовых тарифов + 2 семейных = всего 7 объектов
op.add_tariff(Tariff("Старт",     300,  100,  5))
op.add_tariff(Tariff("Стандарт",  500,  400, 15))
op.add_tariff(Tariff("Премиум",   900, 2000, 50))
op.add_tariff(Tariff("Безлимит", 1500, 5000, 200))
op.add_tariff(Tariff("Молодёжный", 250, 200, 10))
op.add_tariff(FamilyTariff("Семейный Базовый",   400, 500, 20,
                           numbers_count=3, discount=10))
op.add_tariff(FamilyTariff("Семейный Премиум",   800, 1500, 60,
                           numbers_count=4, discount=15))

for t in op:
    print("  ", t)
print(f"  Всего тарифов в линейке: {len(op)}")

print("\n" + "=" * 60)
print("2. Поиск тарифа по названию (find_tariff)")
print("=" * 60)
found = op.find_tariff("премиум")  # регистр не важен
print(f"  Найдено по запросу 'премиум': {found}")
missing = op.find_tariff("Космос")
print(f"  Найдено по запросу 'Космос':  {missing}")

print("\n" + "=" * 60)
print("3. Фильтрация: тарифы дешевле 600 ₽/мес")
print("=" * 60)
cheap = op.affordable(600)
for t in cheap:
    print(f"   {t}")

print("\n" + "=" * 60)
print("4. Сортировка по эффективной цене (по возрастанию)")
print("=" * 60)
for t in op.sorted_by_price():
    print(f"   {t.effective_fee():>7.0f} ₽  —  {t.name}")

print("\n" + "=" * 60)
print("5. Сводная статистика линейки (summary)")
print("=" * 60)
s = op.summary()
print(f"   Кол-во тарифов:       {s['count']}")
print(f"   Средняя плата:        {s['avg_fee']:.2f} ₽")
print(f"   Среднее ГБ в пакете:  {s['avg_gigabytes']:.2f}")
print(f"   Минимальная плата:    {s['min_fee']:.2f} ₽")
print(f"   Максимальная плата:   {s['max_fee']:.2f} ₽")

print("\n" + "=" * 60)
print("6. Удаление тарифа (remove_tariff)")
print("=" * 60)
removed = op.remove_tariff("Молодёжный")
print(f"   Удалён: {removed}")
print(f"   Осталось тарифов: {len(op)}")

## 5. Обработка исключений

Четыре сценария ошибок: неверный тип поля, значение вне диапазона, нарушение уникальности, обращение к несуществующему элементу. В GUI каждое из них превратится в `messagebox.showerror`.


In [ ]:
print("=" * 60)
print("7. Обработка исключений")
print("=" * 60)

# (а) TypeError — неверный тип поля при создании
try:
    print("   Попытка: Tariff(name=123, ...)")
    bad = Tariff(name=123, monthly_fee=500, minutes=100, gigabytes=10)
except TypeError as e:
    print(f"   Ожидаемый TypeError: {e}")

# (б) ValueError — значение вне допустимого диапазона
try:
    print("   Попытка: FamilyTariff(..., discount=150)")
    bad = FamilyTariff("Плохой", 500, 100, 10,
                       numbers_count=3, discount=150)
except ValueError as e:
    print(f"   Ожидаемый ValueError: {e}")

# (в) Дубль названия — ValueError из контейнера
try:
    print("   Попытка добавить тариф 'Старт' второй раз...")
    op.add_tariff(Tariff("Старт", 999, 999, 99))
except ValueError as e:
    print(f"   Ожидаемый ValueError: {e}")

# (г) Удаление несуществующего
try:
    print("   Попытка удалить несуществующий тариф 'Космос'...")
    op.remove_tariff("Космос")
except ValueError as e:
    print(f"   Ожидаемый ValueError: {e}")

print("\n--- Демонстрация завершена ---")

---

# Подготовка к следующему занятию: GUI на Python

Эта модель — фундамент. На следующем занятии мы прикрутим к ней графический интерфейс. Заранее установите нужные библиотеки, чтобы не тратить время на настройку в начале пары.

## Что должно быть установлено

### 1. Python 3.10 или новее

Проверьте версию в терминале / командной строке:

```bash
python --version
```

Если версия ниже 3.10 или Python вообще не установлен — скачайте с официального сайта [python.org/downloads](https://www.python.org/downloads/).

> **Важно для Windows:** при установке поставьте галочку **«Add Python to PATH»** на первом экране установщика.

### 2. Tkinter — основная библиотека для GUI на занятии

`tkinter` входит в стандартную поставку Python и **не требует отдельной установки через `pip`**. Проверьте, что он работает:

```bash
python -m tkinter
```

Если открылось маленькое окно с надписью *«This is a Tcl/Tk widget»* — всё в порядке.

**Если выдало ошибку** `ModuleNotFoundError: No module named '_tkinter'`:

*  **Windows / macOS (установщик с python.org):** переустановите Python и убедитесь, что в установщике отмечена опция *«tcl/tk and IDLE»*.
*  **Linux (Ubuntu/Debian):** `sudo apt install python3-tk`
*  **Linux (Fedora):** `sudo dnf install python3-tkinter`
*  **macOS (Homebrew):** `brew install python-tk`

### 3. (Опционально) PyQt6 — для тех, кто хочет более «взрослый» GUI

Понадобится только если будете делать более сложный вариант интерфейса. Для базового задания достаточно `tkinter`.

```bash
pip install PyQt6
```

### 4. IDE — среда разработки

Любая на выбор:

*  **PyCharm Community Edition** (бесплатная) — [jetbrains.com/pycharm/download](https://www.jetbrains.com/pycharm/download/) — рекомендуется, лучше всего подходит для отладки GUI.
*  **VS Code** — [code.visualstudio.com](https://code.visualstudio.com/) + расширение *Python* от Microsoft.
*  **IDLE** — идёт со стандартным Python, для простых задач достаточно.

## Проверочный скрипт

Запустите этот код локально (в Jupyter он окно не откроет — нужен обычный `.py`-файл). Если появилось окно с кнопкой и при нажатии меняется надпись — всё готово к следующему занятию:

```python
import tkinter as tk
from tkinter import messagebox

def on_click():
    label.config(text="Tkinter работает! ✓")
    messagebox.showinfo("Готово", "Окружение настроено корректно.")

root = tk.Tk()
root.title("Проверка окружения")
root.geometry("320x140")

label = tk.Label(root, text="Нажмите кнопку для проверки",
                 font=("Arial", 11))
label.pack(pady=20)

tk.Button(root, text="Проверить", command=on_click,
          width=15).pack()

root.mainloop()
```

## Что приносить на следующее занятие

1. Свой `.ipynb` или `.py`-файл с **рабочей** моделью предметной области (этот этап).
2. Установленный и проверенный Python с `tkinter`.
3. Краткий план интерфейса: какое будет главное окно, какие кнопки, какая таблица/список. Достаточно эскиза от руки или схемы в любом редакторе.
